In [0]:
dbutils.fs.rm("/Workspace/vehicle_stream_checkpoint", True)

True

In [0]:
from pyspark.sql.functions import *

stream_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

query = stream_df.writeStream \
    .format("memory") \
    .queryName("vehicle_stream") \
    .option("checkpointLocation", "/Workspace/vehicle_stream_checkpoint") \
    .trigger(availableNow=True) \
    .start()

In [0]:
%sql
Select * from vehicle_stream

timestamp,value


In [0]:
from pyspark.sql.functions import *

telemetry_df = stream_df.select(
    concat(lit("V"), (col("value") % 100)).alias("vehicle_id"),
    (rand()*120 + 20).cast("int").alias("speed"),
    (rand()*40 + 80).cast("int").alias("engine_temp"),
    expr("""
        CASE
        WHEN rand() < 0.25 THEN 'Berlin'
        WHEN rand() < 0.5 THEN 'Munich'
        WHEN rand() < 0.75 THEN 'Hamburg'
        ELSE 'Stuttgart'
        END
    """).alias("region"),
    col("timestamp").alias("event_time")
)

In [0]:
bronze_df = telemetry_df.withColumn(
    "event_date",
    to_date("event_time")
)

In [0]:
bronze_query = bronze_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Workspace/bronze_checkpoint") \
    .partitionBy("event_date") \
    .trigger(availableNow=True) \
    .table("bronze_vehicle_events")

In [0]:
%sql
DESCRIBE DETAIL bronze_vehicle_events

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,42b99b6d-3481-4131-927f-323a9f7c611b,workspace.default.bronze_vehicle_events,null,,2026-03-09T17:15:36.905Z,2026-03-11T16:41:34.000Z,List(event_date),List(),5,4552827,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-453466f1-533d-4c1a-be9d-e0e065f30c37, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-d08ac3c0-de34-4353-bdd3-4a637e4d8482)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
SELECT * FROM bronze_vehicle_events limit 10

vehicle_id,speed,engine_temp,region,event_time,event_date
V8,137,118,Stuttgart,2026-03-09T17:15:41.078Z,2026-03-09
V16,20,110,Hamburg,2026-03-09T17:15:42.678Z,2026-03-09
V24,69,117,Berlin,2026-03-09T17:15:44.278Z,2026-03-09
V32,95,98,Berlin,2026-03-09T17:15:45.878Z,2026-03-09
V40,90,117,Berlin,2026-03-09T17:15:47.478Z,2026-03-09
V48,30,107,Hamburg,2026-03-09T17:15:49.078Z,2026-03-09
V56,24,106,Munich,2026-03-09T17:15:50.678Z,2026-03-09
V64,129,87,Hamburg,2026-03-09T17:15:52.278Z,2026-03-09
V72,110,80,Munich,2026-03-09T17:15:53.878Z,2026-03-09
V80,75,85,Munich,2026-03-09T17:15:55.478Z,2026-03-09


In [0]:
%sql
SELECT * FROM bronze_vehicle_events where event_date = "2026-03-11" LIMIT 10

vehicle_id,speed,engine_temp,region,event_time,event_date
V4,40,90,Munich,2026-03-11T00:00:00.278Z,2026-03-11
V12,87,90,Hamburg,2026-03-11T00:00:01.878Z,2026-03-11
V20,45,92,Munich,2026-03-11T00:00:03.478Z,2026-03-11
V28,127,110,Munich,2026-03-11T00:00:05.078Z,2026-03-11
V36,98,97,Munich,2026-03-11T00:00:06.678Z,2026-03-11
V44,102,108,Berlin,2026-03-11T00:00:08.278Z,2026-03-11
V52,109,105,Stuttgart,2026-03-11T00:00:09.878Z,2026-03-11
V60,131,95,Berlin,2026-03-11T00:00:11.478Z,2026-03-11
V68,35,112,Hamburg,2026-03-11T00:00:13.078Z,2026-03-11
V76,55,80,Berlin,2026-03-11T00:00:14.678Z,2026-03-11


Silver Layer


In [0]:
bronze_df = spark.readStream.table("bronze_vehicle_events")

In [0]:
clean_df = bronze_df.filter(
    (col("speed") >= 0) &
    (col("speed") <= 160) &
    (col("engine_temp") >= 70) &
    (col("engine_temp") <= 120) &
    (col("vehicle_id").isNotNull())
)

In [0]:
clean_df.count()

861968

In [0]:
silver_query = clean_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Workspace//silver_checkpoint") \
    .partitionBy("event_date") \
    .trigger(availableNow=True) \
    .table("silver_vehicle_events")

In [0]:
from pyspark.sql.functions import *

silver_df = spark.readStream.table("silver_vehicle_events")

In [0]:
 overheat_df = silver_df.filter(col("engine_temp")>100)

In [0]:
gold_overheat_query = overheat_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Workspace/gold_overheat_checkpoint") \
    .trigger(availableNow=True) \
    .table("gold_vehicle_overheating")

In [0]:
%sql
Select * from gold_vehicle_overheating limit 10

vehicle_id,speed,engine_temp,region,event_time,event_date
V20,103,103,Hamburg,2026-03-10T00:00:03.478Z,2026-03-10
V28,123,104,Berlin,2026-03-10T00:00:05.078Z,2026-03-10
V36,31,112,Hamburg,2026-03-10T00:00:06.678Z,2026-03-10
V60,102,113,Berlin,2026-03-10T00:00:11.478Z,2026-03-10
V76,34,103,Munich,2026-03-10T00:00:14.678Z,2026-03-10
V0,115,104,Munich,2026-03-10T00:00:19.478Z,2026-03-10
V16,72,103,Munich,2026-03-10T00:00:22.678Z,2026-03-10
V32,107,112,Berlin,2026-03-10T00:00:25.878Z,2026-03-10
V40,99,105,Munich,2026-03-10T00:00:27.478Z,2026-03-10
V48,118,116,Munich,2026-03-10T00:00:29.078Z,2026-03-10
